In [2]:
include("CRD_STA.jl")
include("Fun.jl")
using NonlinearEigenproblems
using DelimitedFiles

In [ ]:
using LinearAlgebra
using DelimitedFiles
using SparseArrays
using Printf

# 类型别名提高可读性和性能
const Float = Float64
const ComplexF = ComplexF64
const EigenResult = GeneralizedEigen{ComplexF, ComplexF, Matrix{ComplexF}, Vector{ComplexF}}

# 封装所有模拟参数和预计算值
struct SimulationParams
    # 输入参数
    N_cheb::Int
    Mr::Float
    gamma::Float
    sigma::Float
    Ro::Float
    Tw::Float
    
    # 计算参数
    Co::Float
    
    # 基础流变量
    u0::Vector{Float}
    v0::Vector{Float}
    w0::Vector{Float}
    f::Vector{Float}
    q::Vector{Float}
    D::Matrix{Float}
    D2::Matrix{Float}
    x::Matrix{Float}
    
    # 核心矩阵
    F::Matrix{Float}
    G::Matrix{Float}
    H::Matrix{Float}
    T::Matrix{Float}
    
    # 物理场向量
    rho::Matrix{Float}
    lam::Matrix{Float}
    kappa::Matrix{Float}
    
    function SimulationParams(N_cheb::Int, Mr::Float, gamma::Float, sigma::Float, Ro::Float, Tw::Float)
        # 计算不变值
        Co = 2 - Ro - Ro^2
        
        # 计算基础流 (假设这些函数已定义)
        u0, v0, w0, f, q, D, D2, x = baseflow_var(N_cheb, Ro, "cheb")
        
        # 计算温度和焓场 (假设T_ca函数已定义)
        H_mat, T_mat = T_ca(Mr, f, q, w0, gamma, Tw)
        
        # 插值计算 (假设interp函数已定义)
        F_mat, G_mat, H_mat, T_mat, rho, z = interp(u0, v0, H_mat, T_mat, x, N_cheb, "sim")
        
        # 计算粘性系数
        lam = - (2/3) .* T_mat
        kappa = (1/sigma) .* T_mat
        
        # 创建并返回结构体
        new(N_cheb, Mr, gamma, sigma, Ro, Tw, Co, 
            u0, v0, w0, f, q, D, D2, x,
            F_mat, G_mat, H_mat, T_mat,
            rho, lam, kappa)
    end
end

# 主计算函数
function calculate_optimized(
    be0::Float, be1::Float, be_step::Float,
    params::SimulationParams;
    R_init::Int = 190,
    ali_step1::Float = 0.002,
    ali_step2::Float = 0.001,
    max_iter::Int = 20
)
    # 预分配结果矩阵
    data_total = Matrix{Float}(undef, 0, 6)
    R_current = R_init
    
    # 提取不变参数 (避免在循环中重复访问结构体字段)
    const_params = (
        params.F, params.G, params.H, params.rho, params.lam, params.kappa, params.D,
        params.D2,params.T, params.sigma, params.gamma, params.N_cheb, params.Ro, params.Co
    )
    
    # 主循环 - 遍历be值
    be_values = be0:be_step:be1
    total_steps = length(be_values)
    
    @inbounds for (step_idx, be) in enumerate(be_values)
        @printf("Processing be=%.3f (%d/%d)\n", be, step_idx, total_steps)
        
        # 阶段1: 寻找稳定R
        R_stable = find_stable_R(R_current, be, const_params, params.Mr, max_iter)
        R_current = R_stable  # 更新为当前R
        @printf("  Stable R found: %d\n", R_stable)
        
        # 阶段2: 遍历ali值
        data_stage1 = process_ali_range(
            R_stable, be, 0.18:ali_step1:0.4, const_params, params.Mr
        )
        
        if isempty(data_stage1)
            @warn "  No data found for be=$be, R=$R_stable"
            continue
        end
        
        # 提取关键参数进入阶段3
        ali_val = imag(data_stage1[end, 3])
        R0_start = round(Int, real(data_stage1[end, 1]))
        @printf("  Starting R range from %d with ali=%.4f\n", R0_start, ali_val)
        
        # 阶段3: 遍历R值
        data_stage2 = process_R_range(
            R0_start:1:220, ali_val, be, const_params, params.Mr, ali_step2
        )
        
        # 合并结果
        if !isempty(data_stage2)
            real_data = real.(data_stage2)
            data_total = vcat(data_total, real_data)
            
            # 分批保存数据
            if size(data_total, 1) % 10 == 0
                save_data(data_total, "Dataall_$(params.Tw)_$(params.Mr).dat")
            end
        end
    end
    
    # 最终保存
    save_data(data_total, "Dataall_$(params.Tw)_$(params.Mr).dat")
    return data_total
end

# ==================== 辅助函数实现 ====================

# 阶段1: 寻找稳定R值
function find_stable_R(R_start::Int, be::Float, const_params, Mr::Float, max_iter::Int=20)
    R = R_start
    iter_count = 0
    last_val = 0.0
    
    while iter_count < max_iter
        Ma = Mr / R
        al = complex(0.35, 0.18)  # 固定ali=0.18
        
        # 求解特征值问题
        C = solve_eigenproblem(R, Ma, al, be, const_params)
        val = find_relevant_eigenvalues(C)
        
        # 检查终止条件
        if isempty(val) || imag(val[1]) <= -0.0005
            return R
        end
        
        # 更新R值
        R -= 10
        iter_count += 1
        
        # 检查收敛
        if iter_count > 1 && abs(imag(val[1]) - last_val) < 1e-6
            return R
        end
        last_val = imag(val[1])
    end
    
    @warn "Max iterations reached in find_stable_R. Returning current R=$R"
    return R
end

# 处理ali范围
function process_ali_range(
    R::Int,
    be::Float,
    ali_range::StepRangeLen{Float},
    const_params::Tuple,
    Mr::Float
)
    data_all = Matrix{ComplexF}(undef, 0, 4)
    Ma = Mr / R
    
    for ali in ali_range
        al = complex(0.35, ali)
        C = solve_eigenproblem(R, Ma, al, be, const_params)
        val = find_relevant_eigenvalues(C, imag_max=0.03, real_max=0.05)
        
        if isempty(val)
            data_all = [data_all; [R  be  -1  -1]]
            continue
        end
        

    total, alpha = track_eigenvalues(
        val::Vector{ComplexF}, R::Int, Ma::Float, be::Float, 
        const_params; alr_range=0.35:0.0015:0.5, max_tracks = min(3,length(val)), C)
        
        pinpoints = find_pinpoints(total, alpha, 0.0015)
        
        if !isempty(pinpoints)
            max_idx = argmax(imag.(getindex.(pinpoints, 4)))
            pt = pinpoints[max_idx]
            data_all = [data_all; [pt[1], pt[2], complex(pt[3], ali), pt[4]]]
            if should_terminate(data_all)
                save_data(real.(data_all), "AS.dat")
                return data_all
            end
        else
            data_all = [data_all; [R  be  -1  -1]]
            save_data(real.(data_all), "AS.dat")
        end
    end
    
    save_data(real.(data_all), "AS.dat")
    return data_all
end

# 阶段3: 处理R范围
function process_R_range(R_range, ali::Float, be::Float, const_params, Mr::Float, step_size::Float)
    data_section = Matrix{ComplexF}(undef, 0, 4)
    
    for R in R_range
        Ma = Mr / R
        al = complex(0.35, ali)
        C = solve_eigenproblem(R, Ma, al, be, const_params)
        val = find_relevant_eigenvalues(C, imag_max=0.03, real_max=0.1)
        
        # 跳过无特征值的情况
        if isempty(val)
            push!(data_section, [R, be, -1, -1])
            continue
        end
        
        # 追踪特征值
        total, alpha = track_eigenvalues(
            val, R, Ma, be, const_params,
            alr_range=0.2:step_size:0.5,
            max_tracks=min(2, length(val)),C
        )
        
        # 寻找关键点
        pinpoints = find_pinpoints(total, alpha, step_size)
        
        if isempty(pinpoints)
            push!(data_section, [R, be, -1, -1])
        else
            max_idx = argmax(imag.(getindex.(pinpoints, 4)))
            pt = pinpoints[max_idx]
            push!(data_section, [pt[1], pt[2], complex(pt[3], ali), pt[4]])
            
            # 检查终止条件
            if check_section2_termination(data_section)
                save_data(real.(data_section), "AS1.dat")
                return data_section
            end
        end
    end
    
    save_data(real.(data_section), "AS1.dat")
    return data_section
end

# ==================== 工具函数实现 ====================

# 求解特征值问题
function solve_eigenproblem(R::Int, Ma::Float, al::ComplexF, be::Float, const_params)
    F, G, H, rho, lam, kappa, D, D2,T, sigma, gamma, N_cheb, Ro, Co = const_params
    B0, B1 = Timemode(F, G, H, rho, lam, kappa, T, sigma, gamma, R, Ma, al, be, N_cheb, Ro, Co,D ,D2)
    return eigen(B0, B1)
end

# 查找相关特征值
function find_relevant_eigenvalues(C::EigenResult; imag_max::Float=0.03, real_max::Float=0.08)
    filter(v -> -imag_max < imag(v) < imag_max && abs(real(v)) < real_max, C.values)
end

# 追踪特征值变化
function track_eigenvalues(
    val_init::Vector{ComplexF}, R::Int, Ma::Float, be::Float, 
    const_params; alr_range, max_tracks::Int, C::EigenResult
)
    total = Vector{Vector{ComplexF}}(undef, max_tracks)
    alpha = Vector{Float}(undef, length(alr_range) * max_tracks)
    alpha_idx = 1
    
    for i in 1:max_tracks
        track_vals, track_alphas = track_single_eigenvalue(
            val_init[i], R, Ma, be, const_params, alr_range, C
        )
        total[i] = track_vals
        
        alpha[alpha_idx:alpha_idx+length(track_alphas)-1] = track_alphas
        alpha_idx += length(track_alphas)
    end
    
    return total, alpha[1:alpha_idx-1]
end

# 追踪单个特征值
function track_single_eigenvalue(
    val_start::ComplexF, R::Int, Ma::Float, be::Float, 
    const_params, alr_range, C::EigenResult)
    F, G, H, rho, lam, kappa, D, D2,T, sigma, gamma, N_cheb, Ro, Co = const_params
    val_current = val_start
    vec = eigvector(val_start, C.values, C.vectors)
    values = Vector{ComplexF}(undef, length(alr_range))
    alphas = Vector{Float}(undef, length(alr_range))
    
    for (idx, alr) in enumerate(alr_range)
        al = complex(alr, imag(val_current))  # 保持虚部不变
        B0, B1 = Timemode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,al,be,N_cheb,Ro,Co,D,D2)
        val_new, vec_new = RQI(B0, B1, val_current, q0=vec)
        
        values[idx] = val_new
        alphas[idx] = alr
        
        val_current = val_new
        vec = vec_new
    end
    
    return values, alphas
end
# 寻找关键点
function find_pinpoints(total::Vector{Vector{ComplexF}}, alpha::Vector{Float}, step_size::Float)
    pinpoints = Vector{Vector{Float}}(undef, 0)
    
    for (i, track) in enumerate(total)
        real_vals = real.(track)
        d2 = diff1(real_vals,step_size)
        
        for j in 1:(length(d2)-1)
            if d2[j] * d2[j+1] < 0 && abs(d2[j+1]) < 0.005
                push!(pinpoints, [R, be, alpha[j], track[j]])
            end
        end
    end
    
    return pinpoints
end

# 检查阶段2终止条件
function should_terminate(data::Matrix{ComplexF})
    n = size(data, 1)
    n < 3 && return false
    
    im_vals = imag.(data[:, 4])
    im1 = im_vals[end-2]
    im2 = im_vals[end-1]
    im3 = im_vals[end]
    
    return im2 > im1 && im2 > im3 && im2 != -1
end

# 检查阶段3终止条件
function check_section2_termination(data_section::Matrix{ComplexF})
    n = size(data_section, 1)
    n < 2 && return false
    
    im_current = imag(data_section[end, 4])
    im_prev = imag(data_section[end-1, 4])
    
    return im_prev < 0 && im_current > 0
end

# 保存数据到文件
function save_data(data::AbstractMatrix, filename::String)
    open(filename, "w") do io
        writedlm(io, data)
    end
end

# ==================== 主程序入口 ====================

# 初始化模拟参数
function setup_simulation(;N_cheb=59, Mr=0.3, gamma=1.4, sigma=0.72, Ro=0.2, Tw=1.0)
    return SimulationParams(N_cheb, Mr, gamma, sigma, Ro, Tw)
end

# 运行完整模拟
function run_full_simulation()
    # 设置参数 (包含所有预计算)
    params = setup_simulation()
    
    # 打印参数摘要
    println("Starting simulation with parameters:")
    println("  N_cheb = ", params.N_cheb)
    println("  Mr     = ", params.Mr)
    println("  Ro     = ", params.Ro)
    println("  Tw     = ", params.Tw)
    
    # 运行优化后的计算
    results = calculate_optimized(
        0.1,    # be0
        0.5,    # be1 (小范围测试)
        0.1,    # be_step
        params,
        R_init = 190,
        ali_step1 = 0.002,
        ali_step2 = 0.001,
        max_iter = 10
    )
    
    return results
end

# 执行模拟
if abspath(PROGRAM_FILE) == @__FILE__
    println("Starting CFD simulation...")
    start_time = time()
    
    results = run_full_simulation()
    
    elapsed = time() - start_time
    println("\nSimulation completed in ", round(elapsed, digits=2), " seconds")
    println("Results saved to Dataall_*.dat files")
    
    # 打印结果摘要
    if !isempty(results)
        println("\nSummary of results:")
        println("  Number of data points: ", size(results, 1))
        println("  Max imaginary part: ", maximum(results[:, 6]))
        println("  Min imaginary part: ", minimum(results[:, 6]))
    end
end

In [4]:
params = setup_simulation(
    N_cheb=59,
    Mr=0.3,
    gamma=1.4,
    sigma=0.72,
    Ro=0.2,
    Tw=1.0
)

/home/zhj/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/sN1tQ/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/sN1tQ/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/sN1tQ/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/sN1tQ/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/sN1tQ/src/integr

SimulationParams(59, 0.3, 1.4, 0.72, 0.2, 1.0, 1.76, [6.217769485991772e-26, -0.0015286484682360845, -0.0030528924369473413, -0.004572737849468254, -0.006088190651098306, -0.0075992567890720685, -0.009105942212529361, -0.010608252872485478, -0.012106194721801486, -0.013599773715154587  …  1.9098915920541236e-14, 1.697928026346526e-14, 1.4859041564350273e-14, 1.2738194129484982e-14, 1.0616732171418036e-14, 8.494649808956406e-15, 6.371941157139856e-15, 4.248600417247396e-15, 2.1246219668268163e-15, 4.811533476785787e-32], [1.104405267293563e-27, 0.0014999334555247153, 0.0029998608572470954, 0.004499776165470122, 0.005999673355873761, 0.007499546419503226, 0.008999389362757192, 0.01049919620737596, 0.011998960990429576, 0.0134986777643059  …  0.9999999999999897, 0.9999999999999908, 0.9999999999999919, 0.9999999999999931, 0.9999999999999942, 0.9999999999999953, 0.9999999999999966, 0.9999999999999977, 0.9999999999999989, 1.0], [7.702693512787828e-27, 2.294189279458345e-6, 9.167944707059495e

In [34]:
results = calculate_optimized(
    0.1,
    0.1,
    0.1,
    params,
    R_init=190,
    ali_step1=0.002,
    ali_step2=0.001,
    max_iter=20
)

Processing be=0.100 (1/1)
  Stable R found: 180


InterruptException: InterruptException: